In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split

file_path = '/Users/melina/Downloads/finalpaperdataset.csv'
df = pd.read_csv(file_path)


X = df.drop(columns='Decision')
y = df['Decision']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    test_size=0.30,      
    random_state=42, 
    stratify=y           
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.50,       
    random_state=42, 
    stratify=y_temp
)


print(f"Train set shape: {X_train.shape}, Validation set shape: {X_val.shape}, Test set shape: {X_test.shape}")
print(f"Train target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Validation target distribution:\n{y_val.value_counts(normalize=True)}")
print(f"Test target distribution:\n{y_test.value_counts(normalize=True)}")


Train set shape: (9100, 17), Validation set shape: (1950, 17), Test set shape: (1950, 17)
Train target distribution:
Decision
1    0.536264
0    0.463736
Name: proportion, dtype: float64
Validation target distribution:
Decision
1    0.53641
0    0.46359
Name: proportion, dtype: float64
Test target distribution:
Decision
1    0.53641
0    0.46359
Name: proportion, dtype: float64


Logistic regression

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LogisticRegression

X = df.drop(columns='Decision')
y = df['Decision']

kf = KFold(n_splits=9, shuffle=True, random_state=42)

lr = LogisticRegression(max_iter=1000, random_state=42)
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear'],
    'penalty': ['l2'],
    'class_weight': [None, 'balanced']
}

grid_search = GridSearchCV(
    estimator=lr,
    param_grid=param_grid,
    cv=kf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\n Best Hyperparameters (Train CV only):")
print(grid_search.best_params_)
print("Best Train CV AUC:", round(grid_search.best_score_, 4))


Fitting 9 folds for each of 20 candidates, totalling 180 fits

 Best Hyperparameters (Train CV only):
{'C': 100, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'lbfgs'}
Best Train CV AUC: 0.7326


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix

best_params = {
    'C': 100,
    'penalty': 'l2',
    'solver': 'lbfgs',
    'class_weight': None,
    'random_state': 42
}

lr = LogisticRegression(**best_params)


lr.fit(X_train, y_train)


y_test_pred = lr.predict(X_test)
y_test_proba = lr.predict_proba(X_test)[:, 1]  

print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_test_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_test_pred))


Test Accuracy: 0.6907692307692308

Classification Report:
               precision    recall  f1-score   support

           0       0.66      0.70      0.68       904
           1       0.72      0.69      0.70      1046

    accuracy                           0.69      1950
   macro avg       0.69      0.69      0.69      1950
weighted avg       0.69      0.69      0.69      1950

Test ROC-AUC: 0.7464566870843838

Confusion Matrix:
 [[629 275]
 [328 718]]


/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Decision tree

In [16]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix


dt = DecisionTreeClassifier(random_state=42)

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 8, 12, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8]
}

kf = KFold(n_splits=9, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    cv=kf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\n🔍 Best Hyperparameters:", grid_search.best_params_)
print("Best Train CV AUC:", round(grid_search.best_score_, 4))



Fitting 9 folds for each of 160 candidates, totalling 1440 fits

🔍 Best Hyperparameters: {'criterion': 'gini', 'max_depth': 8, 'min_samples_leaf': 2, 'min_samples_split': 20}
Best Train CV AUC: 0.7377


In [17]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix

best_dt_params = {
    'criterion': 'gini',
    'max_depth': 8,
    'min_samples_leaf': 8,
    'min_samples_split': 20,
    'random_state': 42
}

dt = DecisionTreeClassifier(**best_dt_params)

dt.fit(X_train, y_train)

y_test_pred = dt.predict(X_test)
y_test_proba = dt.predict_proba(X_test)[:, 1]  

print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_test_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_test_pred))


Test Accuracy: 0.6738461538461539

Classification Report:
               precision    recall  f1-score   support

           0       0.63      0.71      0.67       904
           1       0.72      0.64      0.68      1046

    accuracy                           0.67      1950
   macro avg       0.68      0.68      0.67      1950
weighted avg       0.68      0.67      0.67      1950

Test ROC-AUC: 0.7369593817154267

Confusion Matrix:
 [[644 260]
 [376 670]]


Random forest

In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix


rf = RandomForestClassifier(random_state=42)

param_dist = {
    'n_estimators': [100, 200, 300],
    'criterion': ['gini', 'entropy'],
    'max_depth': [5, 8, 12, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2']
}

kf = KFold(n_splits=9, shuffle=True, random_state=42)  

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,           
    cv=kf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

random_search.fit(X_train, y_train)

print("\n🔍 Best Hyperparameters:", random_search.best_params_)
print("Best Train CV AUC:", round(random_search.best_score_, 4))

best_rf = random_search.best_estimator_

y_test_pred = best_rf.predict(X_test)
y_test_proba = best_rf.predict_proba(X_test)[:, 1]

print("\nTest Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_test_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_test_pred))


Fitting 9 folds for each of 50 candidates, totalling 450 fits



🔍 Best Hyperparameters: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None, 'criterion': 'gini'}
Best Train CV AUC: 0.79

Test Accuracy: 0.7317948717948718

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.72      0.71       904
           1       0.75      0.74      0.75      1046

    accuracy                           0.73      1950
   macro avg       0.73      0.73      0.73      1950
weighted avg       0.73      0.73      0.73      1950

Test ROC-AUC: 0.799048524509721

Confusion Matrix:
 [[649 255]
 [268 778]]


SVM(RBF)

In [19]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix


svm_rbf = SVC(
    kernel='rbf',
    probability=True,   
    random_state=42
)

param_grid = {
    'C': [0.5, 1, 5, 10],
    'gamma': ['scale', 0.1, 0.01]
}

kf = KFold(n_splits=9, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=svm_rbf,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=kf,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\n🔍 Best Hyperparameters:", grid_search.best_params_)
print("Best Train CV AUC:", round(grid_search.best_score_, 4))

best_svm = grid_search.best_estimator_

y_test_pred = best_svm.predict(X_test)
y_test_proba = best_svm.predict_proba(X_test)[:, 1]

print("\nTest Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_test_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_test_pred))


Fitting 9 folds for each of 12 candidates, totalling 108 fits



🔍 Best Hyperparameters: {'C': 10, 'gamma': 0.1}
Best Train CV AUC: 0.7359

Test Accuracy: 0.6964102564102564

Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.68      0.68       904
           1       0.72      0.71      0.71      1046

    accuracy                           0.70      1950
   macro avg       0.70      0.70      0.70      1950
weighted avg       0.70      0.70      0.70      1950

Test ROC-AUC: 0.7381210976497065

Confusion Matrix:
 [[619 285]
 [307 739]]


Knn

In [20]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix

knn = KNeighborsClassifier()

param_grid = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]  
}

kf = KFold(n_splits=9, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=kf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\n🔍 Best Hyperparameters:", grid_search.best_params_)
print("Best Train CV AUC:", round(grid_search.best_score_, 4))

best_knn = grid_search.best_estimator_

y_test_pred = best_knn.predict(X_test)
y_test_proba = best_knn.predict_proba(X_test)[:, 1]

print("\nTest Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_test_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_test_pred))


Fitting 9 folds for each of 16 candidates, totalling 144 fits

🔍 Best Hyperparameters: {'n_neighbors': 9, 'p': 1, 'weights': 'distance'}
Best Train CV AUC: 0.7451

Test Accuracy: 0.6892307692307692

Classification Report:
               precision    recall  f1-score   support

           0       0.65      0.70      0.68       904
           1       0.72      0.68      0.70      1046

    accuracy                           0.69      1950
   macro avg       0.69      0.69      0.69      1950
weighted avg       0.69      0.69      0.69      1950

Test ROC-AUC: 0.752665548486438

Confusion Matrix:
 [[631 273]
 [333 713]]


Xgboost

In [21]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 8],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.7, 1.0]
}

kf = KFold(n_splits=9, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=kf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\n🔍 Best Hyperparameters:", grid_search.best_params_)
print("Best Train CV AUC:", round(grid_search.best_score_, 4))

best_xgb = grid_search.best_estimator_

y_test_pred = best_xgb.predict(X_test)
y_test_proba = best_xgb.predict_proba(X_test)[:, 1]

print("\nTest Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_test_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_test_pred))


Fitting 9 folds for each of 72 candidates, totalling 648 fits


/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [04:38:52] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [04:38:52] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [04:38:52] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [04:38:52] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i


🔍 Best Hyperparameters: {'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 8, 'n_estimators': 200, 'subsample': 0.7}
Best Train CV AUC: 0.7829

Test Accuracy: 0.7271794871794872

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.71      0.71       904
           1       0.75      0.75      0.75      1046

    accuracy                           0.73      1950
   macro avg       0.73      0.73      0.73      1950
weighted avg       0.73      0.73      0.73      1950

Test ROC-AUC: 0.7925446073537623

Confusion Matrix:
 [[638 266]
 [266 780]]


Lightmbm

In [22]:
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix

lgbm = lgb.LGBMClassifier(random_state=42)

param_grid = {
    'n_estimators': [100],       
    'max_depth': [5, 8],        
    'learning_rate': [0.1],      
    'num_leaves': [31],          
    'subsample': [1.0],          
    'colsample_bytree': [1.0]    
}

kf = KFold(n_splits=9, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    cv=kf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)


grid_search.fit(X_train, y_train)

print("\n🔍 Best Hyperparameters:", grid_search.best_params_)
print("Best Train CV AUC:", round(grid_search.best_score_, 4))

best_lgbm = grid_search.best_estimator_

y_test_pred = best_lgbm.predict(X_test)
y_test_proba = best_lgbm.predict_proba(X_test)[:, 1]

print("\nTest Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_test_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_test_pred))


Fitting 9 folds for each of 2 candidates, totalling 18 fits
[LightGBM] [Info] Number of positive: 4345, number of negative: 3744
[LightGBM] [Info] Number of positive: 4340, number of negative: 3748
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033997 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1475
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.032549 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1462
[LightGBM] [Info] Number of data points in the train set: 8089, number of used features: 17
[LightGBM] [Info] Number of data points in the train set: 8088, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.536597 -> initscore=0.146652
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.537149 -> initscore=0.148871
[LightGBM] [Info] Number of positive: 4328, number of negative: 3761